<table>
<tr><td><img style="height: 150px;" src="images/geo_hydro1.jpg"></td>
<td bgcolor="#FFFFFF">
    <p style="font-size: xx-large; font-weight: 900; line-height: 100%">AG Dynamics of the Earth</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Juypter notebooks</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Georg Kaufmann</p>
    </td>
</tr>
</table>

# 05: Linear regression explained
----
*Georg Kaufmann,
Geophysics Section,
Institute of Geological Sciences,
Freie Universität Berlin,
Germany*

In [ ]:
import numpy as np
import scipy.optimize
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib import cm
from sklearn.metrics import r2_score

## Preparation: Create color bars

Define different color maps ...

In [ ]:
# color bars
n_colors = 9
mycolors_bold = ['#FF0000', '#0000FF', '#00FF00']
mycolors_light = ['#FFAAAA','#AAAAFF', '#AAFFAA']
cmap_bold = ListedColormap(mycolors_bold)
cmap_light = ListedColormap(mycolors_light)
cmap_set1 = cm.get_cmap('Set1', 9) # Set1,Set2,Set3

## Preparation: Create color bars and random points 

- Create random points $X_{ij}, i=1,n; j=1,1$ in the $x_1$-direction.
Note, that the definition of $X_{ij}$ as matrix is not really needed, but we
want to keep consistency to the `scikit-learn` package ...

- Calculate straight line (intercept $a$, slope $b$) with additional noise,
$y_i(X_i)$. 

The $X_{ij}$ are our **features**, the $y_i$ our **labels**.

In [ ]:

# create sample data
np.random.seed(112)
n=60
a = 0.2
b = 0.5
# features
X=np.zeros(n).reshape(n,1)
X[:,0] = np.random.uniform(-1,1,n)

# labels
y=a + b*X[:,0] + np.random.uniform(-0.2,0.2,n)

fig,ax = plt.subplots(1, 1,figsize=(6,6))
plt.xlim([-1.1,1.1]);plt.xlabel('x$_1$')
plt.ylim([-1.1,1.1]);plt.ylabel('y')
plt.plot(X,y,linewidth=0,marker='o')

----
## Method: Linear regression

The **prediction** in this case is a straight line:
$$
p_i(a_j) = a_0 + a_1 x_1
$$
with $i=1,n$ the number of data points (labels), and $j=1,m$ the number of model coefficients,
in this case $j=2$.

The **cost function** is
$$
\chi^2 = \frac{1}{n} \sum\limits_{i=1}^{n} \left( p_i - y_i \right)^2
$$

Minimisation of the cost function is achieved by calcuating the **partial derivatives**
with respect to the model coefficients $a_j$ and setting them to zero (use $p_i$):
$$
\begin{array}{rcl}
\frac{\partial \chi}{\partial a_0} &=& \frac{2}{n} \sum\limits_{i=1}^{n} \left( p_i - y_i \right) \\
\frac{\partial \chi}{\partial a_1} &=& \frac{2}{n} \sum\limits_{i=1}^{n} \left( p_i - y_i \right) x_i
\end{array}
$$

**NOTE:** There is a smart way to find $a_0$ and $a_1$, we simply set the two derivatives to zero, obtain
a system of two linear equations, and solve this  **analytically** for $a_0$ and $a_1$!

**BUT:** We want to use brute-force numerics ... (and the numerical approach can easily be extended
into $n$ dimensions, $X_{ij},j=1,m$).

We start with an initial guess of the model coefficients, then improve by iteration, 
using the **linear regression** method:

1. Load data set with $n$ points.
2. Choose starting model coefficients, $a_0^{(1)},a_1^{(1)}$.
3. Loop, until convergence achieved:
- Calculate the **cost** function:
    $$
    \chi = \frac{1}{n} \sum\limits_{i=1}^{n} \left( a_0 + a_1 x_i - y_i \right)^2
    $$
    - Calculate the **gradient** of the cost function:
    $$
    \begin{array}{rcl}
\frac{\partial \chi}{\partial a_0} &=& \frac{2}{n} \sum\limits_{i=1}^{n} \left( a_0 + a_1 x_i - y_i \right) \\
\frac{\partial \chi}{\partial a_1} &=& \frac{2}{n} \sum\limits_{i=1}^{n} \left( a_0 + a_1 x_i - y_i \right) x_i
\end{array}
    $$
    - Improve the **coefficient vector**:
    $$
    a_j = a_j - \alpha \frac{\partial \chi}{\partial a_j}
    $$
    with $\alpha$ a learning rate.
   

In [ ]:
n=len(X)
# initialize model coefficients
a_0 = 0.
a_1 = 0.
# learning rate
alpha = 0.01
# loop to improve model coefficients
i=1; imax = 1000
cost_old = 1.e4
loop = True
while loop:
    y_pred = a_0 + a_1 * X[:,0]
    cost   = np.sum((y_pred - y)**2) / float(n)
    a_0    = a_0 - alpha * 2. / float(n) * np.sum(y_pred - y)
    a_1    = a_1 - alpha * 2. / float(n) * np.sum((y_pred - y)*X[:,0])
    if (np.abs(cost-cost_old) < 1.e-8):
        loop = False
    if (i > imax):
        loop = False
    cost_old = cost
    i = i +1
    
print ('Original coefficients:   ',a,b)
print ('Calculated coefficients: ',round(a_0,2),round(a_1,2),cost,i)
print ('R2-score:                ',round(100.*r2_score(y,y_pred),2))

In [ ]:
fig,ax = plt.subplots(1, 1,figsize=(6,6))
plt.xlim([-1.1,1.1]);plt.xlabel('x$_1$')
plt.ylim([-1.1,1.1]);plt.ylabel('y')
plt.plot(X,y,linewidth=0,marker='o')
plt.plot(X[:,0],a + b*X[:,0],linewidth=5,color='grey',label='data')
plt.plot(X[:,0],a_0 + a_1*X[:,0],linewidth=4,color='red',label='model')
plt.legend()

Compare to `LinearRegression`from `scikit-searn` package:

In [ ]:
from sklearn.linear_model import LinearRegression 

clf = LinearRegression(normalize=True)
clf.fit(X,y)
y_pred = clf.predict(X)
print ('R2-score:                ',round(100.*r2_score(y,y_pred),2))
#print(clf.score(X,y))

In [ ]:
fig,ax = plt.subplots(1, 1,figsize=(6,6))
plt.xlim([-1.1,1.1]);plt.xlabel('x$_1$')
plt.ylim([-1.1,1.1]);plt.ylabel('y')
plt.plot(X,y,linewidth=0,marker='o')
plt.plot(X[:,0],a + b*X[:,0],linewidth=5,color='grey',label='data')
plt.plot(X[:,0],a_0 + a_1*X[:,0],linewidth=4,color='red',label='model')
plt.plot(X[:,0],y_pred,linewidth=2,color='green',label='ML')
plt.legend()

... done